### Dataset: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics import confusion_matrix
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 255)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
data_path = "/content/IMDB Dataset.csv"

In [19]:
df = pd.read_csv(data_path)

In [20]:
df.head()

,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of v...",positive
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen-...",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well b...",positive
3,"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situ...",positive


In [21]:
df.shape

(50000, 2)

In [22]:
df = df.iloc[:10000]

In [23]:
df.head()

,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of v...",positive
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen-...",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well b...",positive
3,"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situ...",positive


In [24]:
df.shape

(10000, 2)

In [25]:
df["review"][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [26]:
df["sentiment"].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [29]:
## MISSING VALUES

df.isnull().sum()

,0
review,0
sentiment,0


In [30]:
## DUPLICATE VALUES

df.duplicated().sum()

np.int64(17)

In [32]:
## DROP DUPLICATE VALUES

df.drop_duplicates(inplace = True)
df.duplicated().sum()

np.int64(0)

## Basic Preprocessing
### • Remove tags - HTML
### • Lower case
### • Remove stopwords

In [33]:
import re
def remove_tags(raw_text):
  cleaned_text = re.sub(re.compile("<.*?>"), '', raw_text)
  return cleaned_text

In [34]:
df["review"] = df['review'].apply(remove_tags)

In [36]:
df["review"][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wo

In [38]:
df["review"] = df['review'].apply(lambda x: x.lower())

In [39]:
df["review"][0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other shows wo

In [40]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

sw_list = stopwords.words("english")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [41]:
df["review"] = df["review"].apply(
    lambda x: [item for item in x.split() if item not in sw_list]
).apply(
    lambda x: " ".join(x)
)

In [42]:
df["review"][0]

"one reviewers mentioned watching 1 oz episode hooked. right, exactly happened me.the first thing struck oz brutality unflinching scenes violence, set right word go. trust me, show faint hearted timid. show pulls punches regards drugs, sex violence. hardcore, classic use word.it called oz nickname given oswald maximum security state penitentary. focuses mainly emerald city, experimental section prison cells glass fronts face inwards, privacy high agenda. em city home many..aryans, muslims, gangstas, latinos, christians, italians, irish more....so scuffles, death stares, dodgy dealings shady agreements never far away.i would say main appeal show due fact goes shows dare. forget pretty pictures painted mainstream audiences, forget charm, forget romance...oz mess around. first episode ever saw struck nasty surreal, say ready it, watched more, developed taste oz, got accustomed high levels graphic violence. violence, injustice (crooked guards who'll sold nickel, inmates who'll kill order g

In [47]:
## Independent Variable

X = df.iloc[:,0:1]
y = df['sentiment']

In [48]:
X.head()

,review
0,"one reviewers mentioned watching 1 oz episode hooked. right, exactly happened me.the first thing struck oz brutality unflinching scenes violence, set right word go. trust me, show faint hearted timid. show pulls punches regards drugs, sex violence. ha..."
1,"wonderful little production. filming technique unassuming- old-time-bbc fashion gives comforting, sometimes discomforting, sense realism entire piece. actors extremely well chosen- michael sheen ""has got polari"" voices pat too! truly see seamless edit..."
2,"thought wonderful way spend time hot summer weekend, sitting air conditioned theater watching light-hearted comedy. plot simplistic, dialogue witty characters likable (even well bread suspected serial killer). may disappointed realize match point 2: r..."
3,"basically there's family little boy (jake) thinks there's zombie closet & parents fighting time.this movie slower soap opera... suddenly, jake decides become rambo kill zombie.ok, first going make film must decide thriller drama! drama movie watchable..."
4,"petter mattei's ""love time money"" visually stunning film watch. mr. mattei offers us vivid portrait human relations. movie seems telling us money, power success people different situations encounter. variation arthur schnitzler's play theme, director ..."


In [49]:
y.head()

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive


In [58]:
## It will assign positive with 1 and negative with 0

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()


In [59]:
encoder

LabelEncoder()

In [60]:
y = encoder.fit_transform(y)

In [61]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [62]:
## For data training and testing

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [69]:
X_train.shape

(7986, 1)

In [70]:
X_test.shape

(1997, 1)

In [67]:
y_train

array([1, 1, 0, ..., 0, 0, 1])

In [68]:
y_test

array([1, 1, 0, ..., 1, 0, 0])

In [71]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [72]:
cv

CountVectorizer()

In [73]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [74]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [75]:
X_test_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [77]:
# ML Model
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow,y_train)

GaussianNB()

In [79]:
y_pred = gnb.predict(X_test_bow)

In [80]:
y_pred

array([1, 1, 0, ..., 1, 0, 0])

In [83]:
from sklearn.metrics import accuracy_score,confusion_matrix
accuracy_score(y_test,y_pred)

0.6324486730095142

In [85]:
confusion_matrix(y_test, y_pred)

array([[717, 235],
       [499, 546]])

In [88]:
## Improving Accuracy score using RandomForestClassifier

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

In [89]:
rf

RandomForestClassifier()

In [91]:
rf.fit(X_train_bow,y_train)

RandomForestClassifier()

In [92]:
y_pred = rf.predict(X_test_bow)

In [93]:
y_pred

array([1, 1, 0, ..., 1, 0, 0])

In [94]:
accuracy_score(y_test,y_pred)

0.8497746619929895

In [97]:
# Improving Accracy we choose 3000 frequent words
cv = CountVectorizer(max_features=3000)

In [98]:
cv

CountVectorizer(max_features=3000)

In [99]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()

In [100]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [101]:
X_test_bow = cv.transform(X_test['review']).toarray()

In [102]:
X_test_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [103]:
rf = RandomForestClassifier()

In [104]:
rf

RandomForestClassifier()

In [105]:
rf.fit(X_train_bow,y_train)

RandomForestClassifier()

In [106]:
y_pred = rf.predict(X_test_bow)

In [107]:
y_pred

array([1, 1, 0, ..., 1, 0, 0])

In [108]:
accuracy_score(y_test,y_pred)

0.8352528793189785

## N grams

In [109]:
cv = CountVectorizer(ngram_range=(1,2),max_features=5000)

In [110]:
cv

CountVectorizer(max_features=5000, ngram_range=(1, 2))

In [111]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()

In [112]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [113]:
X_test_bow = cv.transform(X_test['review']).toarray()

In [114]:
X_test_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [115]:
rf = RandomForestClassifier()

In [116]:
rf

RandomForestClassifier()

In [117]:
rf.fit(X_train_bow,y_train)

RandomForestClassifier()

In [118]:
y_pred = rf.predict(X_test_bow)

In [119]:
y_pred

array([0, 1, 0, ..., 1, 0, 0])

In [120]:
accuracy_score(y_test,y_pred)

0.8392588883324987

## TFIDF

In [121]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [122]:
tfidf

TfidfVectorizer()

In [123]:
X_train_tfidf = tfidf.fit_transform(X_train["review"]).toarray()

In [124]:
X_train_tfidf

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [125]:
X_test_tfidf = tfidf.transform(X_test["review"]).toarray()

In [126]:
X_test_tfidf

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [127]:
rf = RandomForestClassifier()

In [128]:
rf

RandomForestClassifier()

In [131]:
rf.fit(X_train_tfidf, y_train)

RandomForestClassifier()

In [132]:
y_pred = rf.predict(X_test_tfidf)

In [133]:
y_pred

array([1, 1, 0, ..., 1, 0, 0])

In [134]:
accuracy_score(y_test, y_pred)

0.8377566349524287

## Word To Vec

In [137]:
!pip install -U gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 27.0 MB/s eta 0:00:00


In [138]:
from gensim.models import Word2Vec
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [139]:
## Tokenize the text

sentences = [text.split() for text in df["review"]]

In [140]:
## Train Word2Vec Model

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [142]:
## Convert each sentence into vector

def get_sentence_vector(sentence):

    words = sentence.split()

    vectors = []

    for word in words:
        if word in w2v_model.wv:
            vectors.append(w2v_model.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

X = np.array(df["review"].apply(get_sentence_vector).tolist())

In [146]:
X

array([[-0.18607761,  0.48062873,  0.00828691, ..., -0.42798406,
         0.07985757,  0.09500572],
       [-0.23012584,  0.43217206,  0.20436959, ..., -0.4155112 ,
         0.06290458,  0.19625585],
       [-0.16994204,  0.5019581 ,  0.06965987, ..., -0.4101901 ,
         0.05127992,  0.14465214],
       ...,
       [-0.2084471 ,  0.636608  ,  0.11626469, ..., -0.6184193 ,
         0.03890821,  0.1162027 ],
       [-0.338009  ,  0.58522403,  0.3053196 , ..., -0.7664419 ,
         0.06457368,  0.1769739 ],
       [-0.1467923 ,  0.3309737 ,  0.03224289, ..., -0.32560456,
         0.12585776,  0.19504437]], dtype=float32)

In [143]:
## Target Column

y = df["sentiment"]

In [147]:
y.head()

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive


In [149]:
## Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [150]:
## Train RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [151]:
rf_model

RandomForestClassifier(random_state=42)

In [152]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [153]:
## Prediction

y_pred = rf_model.predict(X_test)

In [154]:
y_pred

array(['positive', 'negative', 'positive', ..., 'positive', 'positive',
       'negative'], dtype=object)

In [155]:
## Accuracy

accuracy = accuracy_score(y_test, y_pred)

In [156]:
accuracy

0.7516274411617426

In [157]:
print("Accuracy :", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy : 0.7516274411617426

Classification Report:

              precision    recall  f1-score   support

    negative       0.74      0.76      0.75       985
    positive       0.76      0.74      0.75      1012

    accuracy                           0.75      1997
   macro avg       0.75      0.75      0.75      1997
weighted avg       0.75      0.75      0.75      1997

